In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_PATH = '/content/drive/MyDrive/A tfm/data'

In [4]:
df_sales = pd.read_csv(DATA_PATH + '/sales.csv')

In [5]:
df = pd.read_csv(DATA_PATH +'/customer_sociodemographics.csv', index_col=0)

In [6]:
df.head()

,pk_cid,pk_partition,country_id,region_code,gender,age,deceased,salary
0,1375586,2018-01,ES,29.0,H,35,N,87218.10
1,1050611,2018-01,ES,13.0,V,23,N,35548.74
2,1050612,2018-01,ES,13.0,V,23,N,122179.11
3,1050613,2018-01,ES,50.0,H,22,N,119775.54
4,1050614,2018-01,ES,50.0,V,23,N,NaN


In [7]:
df.shape

(5962924, 8)

In [8]:
df['pk_partition'].unique()

array(['2018-01', '2018-02', '2018-03', '2018-04', '2018-05', '2018-06',
       '2018-07', '2018-08', '2018-09', '2018-10', '2018-11', '2018-12',
       '2019-01', '2019-02', '2019-03', '2019-04', '2019-05'],
      dtype=object)

In [9]:
df['pk_cid'].value_counts()

,count
pk_cid,
538227,17
538736,17
539016,17
538902,17
538545,17
...,...
1005736,1
1008380,1
1022690,1


In [10]:
df[df['pk_cid'] == 1375586]

,pk_cid,pk_partition,country_id,region_code,gender,age,deceased,salary
0,1375586,2018-01,ES,29.0,H,35,N,87218.1
1047196,1375586,2018-02,ES,29.0,H,35,N,87218.1
1681276,1375586,2018-03,ES,29.0,H,35,N,87218.1
2299297,1375586,2018-04,ES,29.0,H,35,N,87218.1
2714121,1375586,2018-05,ES,29.0,H,35,N,87218.1
3343336,1375586,2018-06,ES,29.0,H,35,N,87218.1
4581564,1375586,2018-07,ES,29.0,H,35,N,87218.1
5075380,1375586,2018-08,ES,29.0,H,36,N,87218.1
6258167,1375586,2018-09,ES,29.0,H,36,N,87218.1
6572487,1375586,2018-10,ES,29.0,H,36,N,87218.1


In [11]:
df.isnull().sum().sort_values(ascending=False)

,0
salary,1541104
region_code,2264
gender,25
pk_cid,0
country_id,0
pk_partition,0
age,0
deceased,0


## Gender (25 registros pero solamente 2 clientes): moda o valor más aproximado a la media por género por edad

In [12]:
df[df['gender'].isnull()]

,pk_cid,pk_partition,country_id,region_code,gender,age,deceased,salary
1324482,476023,2018-03,ES,28.0,NaN,69,N,89991.42
1959576,476023,2018-04,ES,28.0,NaN,69,N,89991.42
3073415,476023,2018-05,ES,28.0,NaN,69,N,89991.42
3705152,476023,2018-06,ES,28.0,NaN,69,N,89991.42
3854354,476023,2018-07,ES,28.0,NaN,69,N,89991.42
4829286,476023,2018-08,ES,28.0,NaN,69,N,89991.42
5417751,216507,2018-08,ES,28.0,NaN,72,N,104296.62
5525093,476023,2018-09,ES,28.0,NaN,69,N,89991.42
5797798,216507,2018-09,ES,28.0,NaN,72,N,104296.62
6949550,216507,2018-10,ES,28.0,NaN,72,N,104296.62


In [13]:
df[df['pk_cid'] == 216507]

,pk_cid,pk_partition,country_id,region_code,gender,age,deceased,salary
5417751,216507,2018-08,ES,28.0,NaN,72,N,104296.62
5797798,216507,2018-09,ES,28.0,NaN,72,N,104296.62
6949550,216507,2018-10,ES,28.0,NaN,72,N,104296.62
7578659,216507,2018-11,ES,28.0,NaN,72,N,104296.62
8525675,216507,2018-12,ES,28.0,NaN,72,N,104296.62
9164692,216507,2019-01,ES,28.0,NaN,73,N,104296.62
10500564,216507,2019-02,ES,28.0,NaN,72,N,104296.62
11147897,216507,2019-03,ES,28.0,NaN,73,N,104296.62
12090284,216507,2019-04,ES,28.0,NaN,73,N,104296.62
13000310,216507,2019-05,ES,28.0,NaN,73,N,104296.62


In [14]:
df['gender'].value_counts(normalize=True) * 100

,proportion
gender,
H,51.778539
V,48.221461


In [15]:
filtro = df[(df['country_id'] == 'ES') & (df['age'] == 69)]
media_por_genero = filtro.groupby('gender')['salary'].mean()
media_por_genero

,salary
gender,
H,122613.962026
V,141910.174182


In [16]:
import plotly.express as px
df_unicos = df.drop_duplicates(subset='pk_cid')
# Agrupamos por edad y género, y calculamos la media de salario
media_edad_genero = df_unicos.groupby(['age', 'gender'])['salary'].mean().reset_index()

fig = px.line(media_edad_genero, x='age', y='salary', color='gender')
fig.show()

## Region_code: desconocido, solamente tiene region_code 'ES'

In [17]:
df.groupby("country_id")["region_code"].count().sort_values(ascending=False)

,region_code
country_id,
ES,5960660
AT,0
BE,0
BR,0
CA,0
CH,0
CI,0
CL,0
AR,0


In [18]:
df["country_id"].value_counts(normalize=True) * 100

,proportion
country_id,
ES,99.962233
GB,0.007396
FR,0.003773
DE,0.003337
US,0.003270
CH,0.003253
BR,0.001459
BE,0.001358
VE,0.001325


## Salary

In [19]:
df.groupby("country_id")["salary"].count().sort_values(ascending=False)

,salary
country_id,
ES,4421783
RO,9
DZ,7
GB,6
CH,6
CA,4
PE,4
CL,1
AR,0


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5962924 entries, 0 to 13647308
Data columns (total 8 columns):
 #   Column        Dtype  
---  ------        -----  
 0   pk_cid        int64  
 1   pk_partition  object 
 2   country_id    object 
 3   region_code   float64
 4   gender        object 
 5   age           int64  
 6   deceased      object 
 7   salary        float64
dtypes: float64(2), int64(2), object(4)
memory usage: 409.4+ MB


In [21]:
df['salary'].isnull().sum()

np.int64(1541104)

Aquí el código de Yess

In [30]:
df.head()

,pk_cid,pk_partition,country_id,region_code,gender,age,deceased,salary,salary_missing,age_group
0,1375586,2018-01,ES,29.0,H,35,N,87218.10,0,25-35
1,1050611,2018-01,ES,13.0,V,23,N,35548.74,0,<25
2,1050612,2018-01,ES,13.0,V,23,N,122179.11,0,<25
3,1050613,2018-01,ES,50.0,H,22,N,119775.54,0,<25
4,1050614,2018-01,ES,50.0,V,23,N,NaN,1,<25


In [28]:
df['salary_missing'] = df['salary'].isna().astype(int)

In [29]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0,25,35,45,60,120],
    labels=['<25','25-35','35-45','45-60','60+']
)

In [31]:
df['salary'] = df.groupby(
    ['country_id', 'region_code', 'gender', 'age_group']
)['salary'].transform(lambda x: x.fillna(x.median()))

/tmp/ipython-input-482075861.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [32]:
def count_nulls(df, columns):
    """
    Count nulls, non-nulls and % of nulls for a given list of columns.
    Returns a tidy dataframe.
    """
    summary = pd.DataFrame({
        'feature': columns,
        'nulls': df[columns].isnull().sum().values,
        'non_nulls': df[columns].notnull().sum().values,
        'pct_nulls': (df[columns].isnull().mean().values * 100).round(2)
    })
    return summary


In [35]:
df['salary'] = df['salary'].fillna(df['salary'].median())

In [37]:
df.region_code.fillna("sin region").inplace=True

In [38]:
df.gender.fillna("sin genero").inplace=True

In [40]:
df.rename(columns={'salary_missing': 'z_salary_missing', 'age_group': 'z_age_group'}, inplace=True)

In [42]:
cols_to_check = [
    'country_id',
    'region_code',
    'gender',
    'age',
    'deceased',
    'salary',
    'pk_partition',
    'pk_cid',
    'z_salary_missing',
    'z_age_group'
]

count_nulls(df, cols_to_check)

,feature,nulls,non_nulls,pct_nulls
0,country_id,0,5962924,0.00
1,region_code,2264,5960660,0.04
2,gender,25,5962899,0.00
3,age,0,5962924,0.00
4,deceased,0,5962924,0.00
5,salary,0,5962924,0.00
6,pk_partition,0,5962924,0.00
7,pk_cid,0,5962924,0.00
8,z_salary_missing,0,5962924,0.00
9,z_age_group,0,5962924,0.00


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5962924 entries, 0 to 13647308
Data columns (total 10 columns):
 #   Column            Dtype   
---  ------            -----   
 0   pk_cid            int64   
 1   pk_partition      object  
 2   country_id        object  
 3   region_code       float64 
 4   gender            object  
 5   age               int64   
 6   deceased          object  
 7   salary            float64 
 8   z_salary_missing  int64   
 9   z_age_group       category
dtypes: category(1), float64(2), int64(3), object(4)
memory usage: 460.6+ MB


In [46]:
df.to_csv(DATA_PATH+"/clean_customer_data.csv")